In [ ]:
!pip install datasets

In [ ]:
from datasets import load_dataset
database=load_dataset('bitext/Bitext-retail-banking-llm-chatbot-training-dataset')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

bitext-retail-banking-llm-chatbot-traini(…):   0%|          | 0.00/7.87M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25545 [00:00<?, ? examples/s]

In [ ]:
!pip install tiktoken
import tiktoken,os
import numpy as np
from tqdm.auto import tqdm
convertor=tiktoken.get_encoding('gpt2')
dictionary_of_ai='data'
os.makedirs(dictionary_of_ai,exist_ok=True)
if __name__=="__main__":
    def functioning(data):
        ids=convertor.encode_ordinary(data['instruction'])
        store={'ids':ids,'length':len(ids)}
        return store
    def func2(data):
        ids=convertor.encode_ordinary(data['response'])
        store={'ids':ids,'length':len(ids)}
        return store
    if not os.path.exists('tokens.bin'):
        set_tokens=database.map(
            func2,
            remove_columns=['response'],
            desc="tokenizing the data",
            num_proc=8
        )
        for words,nums in set_tokens.items():
            array_len=np.sum(nums['length'],dtype=np.uint64)
            file=os.path.join(dictionary_of_ai,f'{words}.bin')
            datatype=np.uint16
            array=np.memmap(file,dtype=datatype,mode='w+',shape=(array_len,))
            number_of_batches=200
            ind=0
            for batch_ind in tqdm(range(number_of_batches),desc=f'writing {file}'):
                batch=nums.shard(num_shards=number_of_batches,index=batch_ind,contiguous=True)
                array_batch=np.concatenate(batch['ids'])
                array[ind:ind+len(array_batch)]=array_batch
                ind+=len(array_batch)
            array.flush()
            del array

        if not os.path.exists('input.bin'):
            tokens=database.map(
            functioning,
            remove_columns=['instruction'],
            desc="tokenizing the data",
            num_proc=8
        )
        for words,nums in tokens.items():
            array_len=np.sum(nums['length'],dtype=np.uint64)
            file=os.path.join(dictionary_of_ai,'train2.bin')
            datatype=np.uint16
            array=np.memmap(file,dtype=datatype,mode='w+',shape=(array_len,))
            number_of_batches=200
            ind=0
            for batch_ind in tqdm(range(number_of_batches),desc=f'writing {file}'):
                batch=nums.shard(num_shards=number_of_batches,index=batch_ind,contiguous=True)
                array_batch=np.concatenate(batch['ids'])
                array[ind:ind+len(array_batch)]=array_batch
                ind+=len(array_batch)
            array.flush()
            del array

tokenizing the data (num_proc=8):   0%|          | 0/25545 [00:00<?, ? examples/s]

writing data/train.bin:   0%|          | 0/200 [00:00<?, ?it/s]

tokenizing the data (num_proc=8):   0%|          | 0/25545 [00:00<?, ? examples/s]

writing data/train2.bin:   0%|          | 0/200 [00:00<?, ?it/s]

In [ ]:
import numpy as np
import torch

def frame_batch(file):
    data = np.memmap(f'data/{file}.bin', dtype=np.uint16, mode='r')
    number_of_tokens = len(data)

    # Setting hyper-parameters based on file size
    if number_of_tokens < 1000:
        batch_size = 4
        number_of_digits = 64
    elif number_of_tokens < 10000:
        batch_size = 8
        number_of_digits = 128
    else:
        batch_size = 8
        number_of_digits = 128

    # We subtract 1 extra to ensure i + 1 + number_of_digits doesn't overflow
    ix = torch.randint(len(data) - number_of_digits - 1, (batch_size,))

    # x: starts at i, length is number_of_digits
    x_coord = torch.stack([torch.from_numpy((data[i : i+number_of_digits]).astype(np.int64)) for i in ix])

    # y: starts at i+1, length is ALSO number_of_digits
    # This ensures both x and y are (batch_size, 256)
    y_coord = torch.stack([torch.from_numpy((data[i+1 : i+1+number_of_digits]).astype(np.int64)) for i in ix])

    if torch.cuda.is_available():
        device = 'cuda'
        x, y = x_coord.pin_memory().to(device, non_blocking=True), y_coord.pin_memory().to(device, non_blocking=True)
    else:
        device = 'cpu'
        x, y = x_coord.to(device), y_coord.to(device)

    return x, y
# import numpy as np
# import torch
# number_of_digits=0
# def frame_batch(file):
#     data=np.memmap(file,dtype=np.uint16,mode='r')
#     number_of_tokens=len(data)
#     if number_of_tokens<1000:
#         batch_size=4
#         number_of_digits=64
#     elif number_of_tokens<10000:
#         batch_size=8
#         number_of_digits=128
#     else:
#         batch_size=16
#         number_of_digits=256
#     ix=torch.randint(len(data)-number_of_digits-1,(batch_size,))
#     x_coord=torch.stack([torch.from_numpy((data[i:i+number_of_digits]).astype(np.int64)) for i in ix])
#     y_coord=torch.stack([torch.from_numpy((data[i:i+1+number_of_digits]).astype(np.int64))for i in ix])
#     if torch.cuda.is_available():
#         x,y=x_coord.pin_memory().to(device='cuda',non_blocking=True),y_coord.pin_memory().to(device='cuda',non_blocking=True)
#     else:
#         x,y=x_coord.to(device='cpu'),y_coord.to(device='cpu')
#     return x,y

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as func
import math
from dataclasses import dataclass
import numpy as np
from tqdm.auto import tqdm
from contextlib import nullcontext
import os
logits=0
loss=0
class LayerNorm(nn.Module):
    def __init__(self,ndimension,bias):
        super().__init__()
        self.weight=nn.Parameter(torch.ones(ndimension))
        self.bias=nn.Parameter(torch.zeros(ndimension)) if bias else None
    def forward(self,x):
        return func.layer_norm(x,self.weight.shape,self.weight,self.bias,1e-5)
class Attention(nn.Module):
    def __init__(self,config):
        super().__init__()
        assert config.n_embd % config.n_head==0
        self.c_attn=nn.Linear(config.n_embd,3*config.n_embd,bias=config.bias)
        self.c_proj=nn.Linear(config.n_embd,config.n_embd,bias=config.bias)
        self.attn_dropout=nn.Dropout(config.dropout)
        self.resid_dropout=nn.Dropout(config.dropout)
        self.n_head=config.n_head
        self.n_embd=config.n_embd
        self.flash=hasattr(func,'scaled_dot_product_attention')
        if not self.flash:
            self.register_buffer('bias',torch.tril(torch.ones(config.block_size,config.block_size)).view
                                 (1,1,config.block_size,config.block_size))
    def forward(self,x):
        B,T,C=x.size()
        q,k,v=self.c_attn(x).split(self.n_embd,dim=2)
        k=k.view(B,T,self.n_head,C//self.n_head).transpose(1,2)
        q=q.view(B,T,self.n_head,C//self.n_head).transpose(1,2)
        v=v.view(B,T,self.n_head,C//self.n_head).transpose(1,2)
        if self.flash:
            y=func.scaled_dot_product_attention(q,k,v,attn_mask=None,
            dropout_p=self.attn_dropout.p if self.training else 0.0,is_causal=True)
        else:
            att=(q @ k.transpose(-2,1))*(1.0/math.sqrt(k.size(-1)))
            att=att.masked_fill(self.bias[:,:,:T,:T]==0,float('-inf'))
            att=func.softmax(att,dim=1)
            att=self.attn_dropout(att)
            y=att @ v
        y=y.transpose(1,2).contiguous().view(B,T,C)
        y=self.resid_dropout(self.c_proj(y))
        return y
class MLP(nn.Module):
    def __init__(self,config):
        super().__init__()
        self.c_fc=nn.Linear(config.n_embd,4*config.n_embd,bias=config.bias)
        self.gelu=nn.GELU()
        self.c_proj=nn.Linear(4*config.n_embd,config.n_embd,bias=config.bias)
        self.dropout=nn.Dropout(config.dropout)
    def forward(self,x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))
class Block(nn.Module):
    def __init__(self,config):
        super().__init__()
        self.ln1=LayerNorm(config.n_embd,config.bias)
        self.attn=Attention(config)
        self.ln2=LayerNorm(config.n_embd,config.bias)
        self.mlp=MLP(config)
    def forward(self,x):
        x=x+self.attn(self.ln1(x))
        x=x+self.mlp(self.ln2(x))
        return x
@dataclass
class Setting_Up_GPT:
    vocab_size:int
    block_size:int
    n_layer:int
    n_head:int
    n_embd:int
    dropout:float=0.0
    bias:bool=True
class GPT(nn.Module):
    def __init__(self,config):
        super().__init__()
        self.config=config
        self.transformer=nn.ModuleDict(
            dict(wte=nn.Embedding(config.vocab_size,config.n_embd),
                 wpe=nn.Embedding(config.vocab_size,config.n_embd),
                 drop=nn.Dropout(config.dropout),
                 h=nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
                 ln_f=LayerNorm(config.n_embd,config.bias)))
        self.lm_head=nn.Linear(config.n_embd,config.vocab_size,bias=False)
        self.transformer.wte.weight=self.lm_head.weight
        self.apply(self._init_weights)
        for pn,p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                nn.init.normal_(p,mean=0.0,std=0.02/math.sqrt(2*config.n_layer))
    def _init_weights(self,module):
        if isinstance(module,nn.Linear):
            nn.init.normal_(module.weight,mean=0.0,std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module,nn.Embedding):
            nn.init.normal_(module.weight,mean=0.0,std=0.02)
    def forward(self,idx,targets=None):
        device=idx.device
        b,t=idx.size()
        assert t<=self.config.block_size
        pos=torch.arange(0,t,dtype=torch.long,device=device)
        tok_emb=self.transformer.wte(idx)
        pos_emb=self.transformer.wpe(pos)
        x=self.transformer.drop(tok_emb+pos_emb)
        for bl in self.transformer.h:
            x=bl(x)
        x=self.transformer.ln_f(x)
        if targets is not None:
            logits = self.lm_head(x)
            loss = func.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
            return logits, loss
        else:
            logits = self.lm_head(x[:, [-1], :])
            return logits, None
    @torch.no_grad()
    def generate(self,idx,new_tokens,temperature=1.0,top_k=None):
        for _ in range(new_tokens):
            idx_cond=idx if idx.size(1)<=self.config.block_size else idx[:,-self.config.block_size:]
            logits,_=self(idx_cond)
            logits=logits[:,-1,:]/temperature
            if top_k:
                v,_=torch.topk(logits,min(top_k,logits.size(-1)))
                logits[logits<v[:,[-1]]]=-float('inf')
            probability=func.softmax(logits,dim=1)
            idx_next=torch.multinomial(probability,num_samples=1)
            idx=torch.cat((idx,idx_next),dim=1)
        return idx
config=Setting_Up_GPT(
    vocab_size=50257,      # GPT-2 tokenizer vocab size (keep as is)
    block_size=256,        # Context window - 256 tokens is good balance
    n_layer=8,             # 8 transformer layers (moderate depth)
    n_head=8,              # 8 attention heads (must divide n_embd evenly)
    n_embd=512,            # 512 embedding dimensions
    dropout=0.15,          # 15% dropout for regularization
    bias=True              # Keep bias terms
  )
model=GPT(config)

In [ ]:
@torch.no_grad()
def estimate_loss(model):
    out = {}
    model.eval()

    # Evaluate on both datasets
    for split in ['train1', 'train2']:
        losses = torch.zeros(eval_iters)

        for k in range(eval_iters):
            # Map split names to actual file names
            if split == 'train1':
                X, Y = frame_batch('train')  # or 'train.bin' depending on actual filename
            else:
                X, Y = frame_batch('train2')

            X, Y = X.to(device), Y.to(device)

            with ctx:
                logits, loss = model(X, Y)
            losses[k] = loss.item()

        out[split] = losses.mean()

    model.train()
    return out
# def estimate_loss(model):
#     out = {}
#     model.eval()
#     with torch.inference_mode():
#         for split in ['train.bin','train2.bin']:
#             losses = torch.zeros(eval_iters)
#             for k in range(eval_iters):
#               X,Y=frame_batch(split)
#               X,Y=X.to(device),Y.to(device)
#               with ctx:
#                   logits, loss = model(X, Y)
#               losses[k] = loss.item()
#             out[split] = losses.mean()
#     model.train()
#     return out

In [ ]:
import torch
from contextlib import nullcontext
learning_rate=1e-4
max_iters=20000
warmup_steps=1000
min_lr=5e-4
eval_iters=500
batch_size=32
block_size=128
gradient_accumulation=8
device =  "cuda" if torch.cuda.is_available() else "cpu"
device_type = 'cuda' if 'cuda' in device else 'cpu'
dtype='bfloat16' if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else 'float16'
ptdtype={'float32': torch.float32, 'bfloat16': torch.bfloat16, 'float16': torch.float16}[dtype]
ctx = nullcontext() if device_type == 'cpu' else torch.amp.autocast(device_type=device_type, dtype=ptdtype)
torch.set_default_device(device)
torch.manual_seed(42)

In [ ]:
from torch.optim.lr_scheduler import LinearLR,SequentialLR,CosineAnnealingLR
import torch
optimizer=torch.optim.AdamW(model.parameters(),lr=learning_rate,betas=(0.9,0.95),weight_decay=0.1,eps=1e-9)
warup=LinearLR(optimizer,total_iters=warmup_steps)
decay=CosineAnnealingLR(optimizer,T_max=max_iters-warmup_steps,eta_min=min_lr)
scheduler=SequentialLR(optimizer,schedulers=[warup,decay],milestones=[warmup_steps])
scaler = torch.cuda.amp.GradScaler(enabled=(dtype == 'float16'))

/tmp/ipython-input-788366990.py:7: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(dtype == 'float16'))
/usr/local/lib/python3.12/dist-packages/torch/cuda/amp/grad_scaler.py:31: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  super().__init__(


In [ ]:
import numpy as np

# Load both files
data1 = np.memmap('data/train.bin', dtype=np.uint16, mode='r')
data2 = np.memmap('data/train2.bin', dtype=np.uint16, mode='r')

print(f"File 1 (train) size: {len(data1)} tokens")
print(f"File 2 (train2) size: {len(data2)} tokens")

# Decode and print first few tokens to see the content
# You'll need your tokenizer/vocab to decode
# Example (adjust based on your tokenizer):
print("\n--- First 200 tokens from train ---")
print(data1[:200])

print("\n--- First 200 tokens from train2 ---")
print(data2[:200])

File 1 (train) size: 4997545 tokens
File 2 (train2) size: 325674 tokens

--- First 200 tokens from train ---
[   40  1101   994   284  3342   345   351   326     0 13144   803   534
  2657   318   281  1593  2239   284  3599   290 13226   663  4034    13
  3423   338   703   345   460 15155   534  2657    25   198   198    16
    13   406 13369   262 14916  7729    25 23591   319   262  2657 44168
    11   345   743  1064   262 14916  7729   319   257 27735  7223   284
   262  2657  2346    11   287   262  7062  5301    11   393   319   262
 44168   338  3052    13   198   198    17    13 16440   262  2657 44168
   338 14916  3052    25  8554   534  3644   393  5175  3335    11  1280
   257  3992  6444   290 16500   284   262  2657 44168   338  3052    13
  6803   329   262 14916  2443   393  2665    13   198   198    18    13
  6062   262  2672  1321    25  7281   262 36454   319   262 14916  2443
   290  2148   262  3306  1321    13   770   743  2291   534  2657  1271
    11  2614  3

In [ ]:
import numpy as np
import tiktoken

# Initialize tiktoken encoder
enc = tiktoken.get_encoding("gpt2")  # Use "gpt2" or "cl100k_base" depending on what you used

def decode(token_ids):
    """Convert token IDs back to text"""
    return enc.decode(token_ids)

# Load both files
data1 = np.memmap('data/train.bin', dtype=np.uint16, mode='r')
data2 = np.memmap('data/train2.bin', dtype=np.uint16, mode='r')

print(f"File 1 (train) size: {len(data1)} tokens")
print(f"File 2 (train2) size: {len(data2)} tokens")

print("\n--- First 200 tokens from train (raw) ---")
print(data1[:200])

print("\n--- First 200 tokens from train (decoded) ---")
print(decode(data1[:200].tolist()))

print("\n" + "="*50)

print("\n--- First 200 tokens from train2 (raw) ---")
print(data2[:200])

print("\n--- First 200 tokens from train2 (decoded) ---")
print(decode(data2[:200].tolist()))

File 1 (train) size: 4997545 tokens
File 2 (train2) size: 325674 tokens

--- First 200 tokens from train (raw) ---
[   40  1101   994   284  3342   345   351   326     0 13144   803   534
  2657   318   281  1593  2239   284  3599   290 13226   663  4034    13
  3423   338   703   345   460 15155   534  2657    25   198   198    16
    13   406 13369   262 14916  7729    25 23591   319   262  2657 44168
    11   345   743  1064   262 14916  7729   319   257 27735  7223   284
   262  2657  2346    11   287   262  7062  5301    11   393   319   262
 44168   338  3052    13   198   198    17    13 16440   262  2657 44168
   338 14916  3052    25  8554   534  3644   393  5175  3335    11  1280
   257  3992  6444   290 16500   284   262  2657 44168   338  3052    13
  6803   329   262 14916  2443   393  2665    13   198   198    18    13
  6062   262  2672  1321    25  7281   262 36454   319   262 14916  2443
   290  2148   262  3306  1321    13   770   743  2291   534  2657  1271
    11  2

In [ ]:
import torch
from tqdm.auto import tqdm
loss_value=float('inf')
best_model="best_model_params.pt"
train1,train2=[],[]
model=model.to(device)
for epoch in tqdm(range(max_iters)):
    if epoch%eval_iters==0 and epoch!=0:
        losses=estimate_loss(model)
        print(f"Epoch {epoch}: train loss {losses['train1']:.4f},query loss {losses['train2']:.4f}")
        train1+=[losses['train1']]
        train2+=[losses['train2']]
        if losses['train2']<loss_value:
            loss_value=losses['train2']
            torch.save(model.state_dict(),best_model)
    dataset='train2' if epoch%2==0 else 'train'
    X,Y=frame_batch(dataset)
    X,Y=X.to(device),Y.to(device)
    with ctx:
        logits,loss=model(X,Y)
        loss=loss/gradient_accumulation
        scaler.scale(loss).backward()
    if((epoch+1)%gradient_accumulation==0) or (epoch+1==max_iters):
        torch.nn.utils.clip_grad_norm_(model.parameters(),max_norm=0.5)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
    scheduler.step()

NameError: name 'model' is not defined

In [ ]:
import nltk
from nltk.stem import PorterStemmer,WordNetLemmatizer
objects=PorterStemmer()
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
import re
from nltk.corpus import stopwords
data="""
Ships are typically larger than boats, but there is no universally accepted distinction between the two. Ships generally can remain at sea for longer periods of time than boats.[3] A legal definition of ship from Indian case law is a vessel that carries goods by sea.[4] A common notion is that a ship can carry a boat, but not vice versa.[5] A ship is likely to have a full-time crew assigned.[6] A US Navy rule of thumb is that ships heel towards the outside of a sharp turn, whereas boats heel towards the inside[7] because of the relative location of the center of mass versus the center of buoyancy.[8][9] American and British 19th century maritime law distinguished "vessels" from other watercraft; ships and boats fall in one legal category, whereas open boats and rafts are not considered vessels.[10]

Starting around the middle of the 18th century, sailing vessels started to be categorised by their type of rig. (Previously they were described by their hull type – for example pink, cat.) Alongside the other rig types such as schooner and brig,[a] the term "ship" referred to the rig type. In this sense, a ship is a vessel with three or more masts, all of which are square-rigged.
"""
tokenized_content=nltk.sent_tokenize(data)
documents=[]
for tokens in tokenized_content:
  formatted_token=re.sub('[^-a-zA-Z]',' ',tokens)
  documents.append(formatted_token)
documents

[' Ships are typically larger than boats  but there is no universally accepted distinction between the two ',
 'Ships generally can remain at sea for longer periods of time than boats ',
 '    A legal definition of ship from Indian case law is a vessel that carries goods by sea ',
 '    A common notion is that a ship can carry a boat  but not vice versa ',
 '    A ship is likely to have a full-time crew assigned ',
 '    A US Navy rule of thumb is that ships heel towards the outside of a sharp turn  whereas boats heel towards the inside    because of the relative location of the center of mass versus the center of buoyancy ',
 '       American and British   th century maritime law distinguished  vessels  from other watercraft  ships and boats fall in one legal category  whereas open boats and rafts are not considered vessels ',
 '      Starting around the middle of the   th century  sailing vessels started to be categorised by their type of rig ',
 ' Previously they were described by t

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
cv=CountVectorizer(vocabulary=['boats','large','sailing','crew'])
bagging=[]
for sen in documents:
  words=nltk.word_tokenize(sen)
  w0=cv.fit_transform([sen])
  bagging.append(w0.toarray())
bagging

[array([[1, 0, 0, 0]]),
 array([[1, 0, 0, 0]]),
 array([[0, 0, 0, 0]]),
 array([[0, 0, 0, 0]]),
 array([[0, 0, 0, 1]]),
 array([[1, 0, 0, 0]]),
 array([[2, 0, 0, 0]]),
 array([[0, 0, 1, 0]]),
 array([[0, 0, 0, 0]]),
 array([[0, 0, 0, 0]]),
 array([[0, 0, 0, 0]])]

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import one_hot
from sklearn.metrics import mean_squared_error,r2_score,accuracy_score
import pandas as pd
mvp_dataset=pd.read_csv('startup_mvp_dataset.csv')
ideas=mvp_dataset['startup_idea'].values
vocabulory_size=mvp_dataset['startup_idea'].nunique()
dicts={'D2C':0,'B2C':1,'B2B':2,'SaaS':3}
encoded1=pd.DataFrame([one_hot(idea,vocabulory_size)for idea in ideas])
mvp_dataset['sector']=mvp_dataset['sector'].map(dicts)
output=mvp_dataset['months_to_launch']
train = pd.concat([
    mvp_dataset[['sector','monthly_income','monthly_expenses','monthly_savings',
                 'current_savings','investment','survival_buffer','total_capital']],
    encoded1
], axis=1)
train.columns=train.columns.astype(str)
first_model=RandomForestRegressor(n_estimators=300,max_depth=10,min_samples_split=3,min_samples_leaf=4,random_state=42)
X,x,Y,y=train_test_split(train,output,test_size=0.2,random_state=42)
first_model.fit(X,Y)
prediction=first_model.predict(x)
print(f"mean squared error:{mean_squared_error(y,prediction)}")
print(f"r2 score:{r2_score(y,prediction)}")

mean squared error:40.145884070946295
r2 score:0.980352428744251


In [ ]:
from xgboost import XGBRegressor
xg_model=XGBRegressor(n_estimators=300,max_depth=15,learning_rate=0.1,subsample=0.8,colsample_bytree=0.8,
                      random_state=42,eval_metric='rmse')
xg_model.fit(X,Y)
pred=xg_model.predict(x)
acc1=r2_score(y,pred)
acc2=mean_squared_error(y,pred)
print(f'r2 score:{acc1}')
print(f'mean squared error:{acc2}')

r2 score:0.9831430456151006
mean squared error:34.44381637386265


In [ ]:
from sklearn.linear_model import LogisticRegression
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer=TfidfVectorizer()
complex_data = pd.read_csv('complexity_dataset.csv')
encoded2=vectorizer.fit_transform(complex_data['startup_idea'])
X1,x1,Y1,y1=train_test_split(encoded2,complex_data['complexity'],
                             test_size=0.2,random_state=42)
logistic=LogisticRegression(max_iter=1000)
logistic.fit(X1,Y1)
pred2=logistic.predict(x1)
print(f"Logistic Regression:{accuracy_score(pred2,y1)}")

Logistic Regression:0.75


In [ ]:
joblib.dump(logistic,'Logistic_complexity.pkl')
joblib.dump(vectorizer,'tfidfConvertor.pkl')

['tfidfConvertor.pkl']

In [ ]:
import random,joblib
from sklearn.linear_model import LinearRegression
from sklearn.feature_extraction.text import TfidfVectorizer
complex_data = pd.read_csv('complexity_dataset.csv')
sectors_base_cost={
    0:{'tech':(200000, 500000),'legal':(10000, 25000),'marketing':(50000, 150000)},
    1:{'tech':(500000, 1200000),'legal':(15000, 35000),'marketing':(100000, 300000)},
    2:{'tech':(800000, 2000000),'legal':(20000, 50000),'marketing':(80000, 200000)},
    3:{'tech':(1200000, 3500000),'legal':(15000, 40000),'marketing':(100000, 250000)}
}
dicts_complexity={'simple':0.8,'moderate':1.0,'complex':1.3,'advanced':1.5}
convertor=joblib.load('tfidfConvertor.pkl')
logistic_model=joblib.load('Logistic_complexity.pkl')
for index,row in mvp_dataset.iterrows():
  base=sectors_base_cost[row['sector']]
  vector=convertor.transform([row['startup_idea']])
  pre=logistic_model.predict(vector)
  comp=dicts_complexity[pre[0]]
  tech=(base['tech'][0]+base['tech'][1])//2
  legal=(base['legal'][0]+base['legal'][1])//2
  marketing=(base['marketing'][0]+base['marketing'][1])//2
  mvp_dataset.at[index,'mvp_cost'] = int((tech + legal + marketing) * comp*random.uniform(0.8,1.2))
print(mvp_dataset[['startup_idea','sector','mvp_cost']].head())

                 startup_idea  sector  mvp_cost
0   fitness supplements brand       0    388077
1    ayurvedic products brand       0    411730
2  healthcare appointment app       1   1091161
3            ride sharing app       1    876246
4    ayurvedic products brand       0    548427


In [ ]:
train2=pd.concat([mvp_dataset[['sector']],encoded1],axis=1)
output2=mvp_dataset[['mvp_cost']]
train2.columns=train2.columns.astype(str)
model2=RandomForestRegressor(n_estimators=100,max_depth=10,min_samples_split=3,
                             min_samples_leaf=6,random_state=42)
X0,x0,Y0,y0=train_test_split(train2,output2,test_size=0.2,random_state=42)
model2.fit(X0,Y0)
preds=model2.predict(x0)
print(f'R2 score:{r2_score(y0,preds)}')
print(f'Mean squared error:{mean_squared_error(y0,preds)}')

/usr/local/lib/python3.12/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


R2 score:0.9614047387597219
Mean squared error:53485658620.45881


In [ ]:

train2=pd.concat([mvp_dataset[['sector']],encoded1],axis=1)
output2=mvp_dataset[['mvp_cost']]
train2.columns=train2.columns.astype(str)
model2=XGBRegressor(n_estimators=300,max_depth=15,learning_rate=0.1,subsample=0.8,colsample_bytree=0.8,
                      random_state=42,eval_metric='rmse')
X0,x0,Y0,y0=train_test_split(train2,output2,test_size=0.2,random_state=42)
model2.fit(X0,Y0)
preds=model2.predict(x0)
print(f'R2 score:{r2_score(y0,preds)}')
print(f'Mean squared error:{mean_squared_error(y0,preds)}')

R2 score:0.7950259447097778
Mean squared error:222682775552.0


In [ ]:
joblib.dump(model2,'total_cost_predictor.pkl')

['total_cost_predictor.pkl']